In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/input/datasets/vtphatt2")

DATASETS = [
    "genimage-adm",
    "genimage-biggan",
    "genimage-glide",
    "genimage-midjourney-part-1",
    "genimage-stable-diffusion-v1-4",
    "genimage-stable-diffusion-v1-5",
    "genimage-wukong",
]

def show_tree(root, max_depth=6, depth=0):
    if depth > max_depth:
        return

    try:
        entries = sorted(root.iterdir(), key=lambda x: x.name.lower())
    except Exception as e:
        print("  ERROR:", e)
        return

    for p in entries:
        indent = "    " * depth

        if p.is_dir():
            print(f"{indent}📁 {p.name}/")

            # If this is a potential image directory, DON'T enter it.
            if p.name.lower() in {"ai", "nature", "real", "fake"}:
                try:
                    # This operation reads directory metadata only.
                    count = sum(1 for _ in p.iterdir())
                    print(f"{indent}   → {count:,} files")
                except Exception as e:
                    print(f"{indent}   → unable to count: {e}")
            else:
                show_tree(p, max_depth, depth + 1)

        else:
            print(f"{indent}📄 {p.name}")


for dataset in DATASETS:
    path = ROOT / dataset

    print("\n" + "=" * 90)
    print(dataset)
    print("=" * 90)

    if not path.exists():
        print("❌ NOT FOUND")
        continue

    show_tree(path)

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import pandas as pd
import random
import json
import os

ROOT = Path("/kaggle/input/datasets/vtphatt2")
OUT = Path("/kaggle/working/signalscope_audit")
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Exact dataset locations discovered from Step 1
# ------------------------------------------------------------

DATASETS = {
    "ADM": ROOT / "genimage-adm/GenImage/ADM/imagenet_ai_0508_adm",
    "BigGAN": ROOT / "genimage-biggan/BigGAN/imagenet_ai_0419_biggan",
    "GLIDE": ROOT / "genimage-glide/glide/imagenet_glide",
    "SD1.4": ROOT / "genimage-stable-diffusion-v1-4/GenImage/stable_diffusion_v_1_4",
    "SD1.5": ROOT / "genimage-stable-diffusion-v1-5/GenImage/stable_diffusion_v_1_5",
    "Wukong": ROOT / "genimage-wukong/GenImage/wukong",
}

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Number of images to actually open per class/split.
# This is enough for an initial statistical audit.
SAMPLE_PER_FOLDER = 300

random.seed(42)

summary = []
samples = []

print("=" * 100)
print("SIGNALSCOPE — STEP 2 DATASET AUDIT")
print("=" * 100)

# ------------------------------------------------------------
# Audit normal directory-based datasets
# ------------------------------------------------------------

for generator, base in DATASETS.items():

    print(f"\n{'=' * 80}")
    print(f"GENERATOR: {generator}")
    print(f"PATH: {base}")
    print("=" * 80)

    if not base.exists():
        print("❌ PATH NOT FOUND")
        continue

    for split in ["train", "val"]:
        for label_name, label in [("ai", 1), ("nature", 0)]:

            folder = base / split / label_name

            if not folder.exists():
                print(f"⚠️ Missing: {split}/{label_name}")
                continue

            # Directory listing only — no image decoding here
            files = [
                p for p in folder.iterdir()
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS
            ]

            total = len(files)

            print(
                f"{split:5s} | "
                f"{label_name:6s} | "
                f"{total:,} images"
            )

            # Random sample for actual image inspection
            sample_files = (
                random.sample(files, min(SAMPLE_PER_FOLDER, total))
                if total > SAMPLE_PER_FOLDER
                else files
            )

            format_counts = Counter()
            size_counts = Counter()
            corrupted = 0
            exif_present = 0

            for path in sample_files:

                try:
                    with Image.open(path) as img:

                        # Verify image integrity
                        img.verify()

                    # Reopen because verify() closes image state
                    with Image.open(path) as img:

                        width, height = img.size

                        format_counts[path.suffix.lower()] += 1
                        size_counts[(width, height)] += 1

                        if img.getexif():
                            exif_present += 1

                        samples.append({
                            "generator": generator,
                            "split": split,
                            "label": label,
                            "label_name": label_name,
                            "path": str(path),
                            "width": width,
                            "height": height,
                            "format": path.suffix.lower(),
                            "aspect_ratio": round(width / height, 4)
                        })

                except Exception:
                    corrupted += 1

            summary.append({
                "generator": generator,
                "split": split,
                "label": label_name,
                "total_images": total,
                "sampled": len(sample_files),
                "corrupted_in_sample": corrupted,
                "exif_in_sample": exif_present,
                "formats": dict(format_counts),
                "top_resolutions": dict(size_counts.most_common(10)),
            })

# ------------------------------------------------------------
# Print summary
# ------------------------------------------------------------

df_summary = pd.DataFrame(summary)

print("\n\n" + "=" * 100)
print("IMAGE COUNTS")
print("=" * 100)

print(
    df_summary[
        ["generator", "split", "label", "total_images",
         "sampled", "corrupted_in_sample", "exif_in_sample"]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# Aggregate real/fake counts
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("REAL vs AI")
print("=" * 100)

counts = (
    df_summary
    .groupby(["generator", "split", "label"])["total_images"]
    .sum()
    .unstack(fill_value=0)
)

print(counts)

# ------------------------------------------------------------
# Sample-level format/resolution analysis
# ------------------------------------------------------------

df_samples = pd.DataFrame(samples)

print("\n" + "=" * 100)
print("SAMPLE FORMAT DISTRIBUTION")
print("=" * 100)

print(
    pd.crosstab(
        [df_samples.generator, df_samples.split, df_samples.label_name],
        df_samples.format
    )
)

print("\n" + "=" * 100)
print("TOP RESOLUTIONS FROM SAMPLE")
print("=" * 100)

resolution_table = (
    df_samples
    .groupby(["generator", "split", "label_name"])
    .apply(lambda x: x[["width", "height"]].value_counts().head(10), include_groups=False)
)

print(resolution_table)

print("\n" + "=" * 100)
print("ASPECT RATIO SUMMARY")
print("=" * 100)

print(
    df_samples
    .groupby(["generator", "split", "label_name"])["aspect_ratio"]
    .agg(["min", "median", "max"])
    .round(3)
)

# ------------------------------------------------------------
# Save audit results
# ------------------------------------------------------------

df_summary.to_csv(OUT / "audit_summary.csv", index=False)
df_samples.to_csv(OUT / "image_samples.csv", index=False)

with open(OUT / "audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 100)
print("AUDIT FILES SAVED")
print("=" * 100)

print(OUT / "audit_summary.csv")
print(OUT / "audit_summary.json")
print(OUT / "image_samples.csv")

print("\n✅ STEP 2 INITIAL AUDIT COMPLETE")

In [ ]:
# Fix JSON serialization of tuple resolution keys

def make_json_safe(obj):
    if isinstance(obj, dict):
        return {
            str(k): make_json_safe(v)
            for k, v in obj.items()
        }
    elif isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    else:
        return obj


safe_summary = make_json_safe(summary)

with open(OUT / "audit_summary.json", "w") as f:
    json.dump(safe_summary, f, indent=2)

print("✅ audit_summary.json saved")
print(f"📁 {OUT}")

In [ ]:
import pandas as pd
from pathlib import Path

AUDIT_DIR = Path("/kaggle/working/signalscope_audit")

df = pd.read_csv(AUDIT_DIR / "image_samples.csv")

print("=" * 90)
print("SIGNALSCOPE — STEP 3: DATASET BIAS ANALYSIS")
print("=" * 90)

# ---------------------------------------------------------
# 1. Format vs label
# ---------------------------------------------------------

print("\n[1] FORMAT × LABEL")
print("-" * 60)

print(
    pd.crosstab(
        df["label_name"],
        df["format"],
        normalize="index"
    ).round(3)
)

# ---------------------------------------------------------
# 2. Resolution vs label
# ---------------------------------------------------------

print("\n[2] RESOLUTION × LABEL")
print("-" * 60)

resolution = (
    df.groupby(["generator", "label_name"])
      .apply(lambda x: x[["width", "height"]]
             .value_counts()
             .head(5))
)

print(resolution)

# ---------------------------------------------------------
# 3. Aspect ratio
# ---------------------------------------------------------

print("\n[3] ASPECT RATIO")
print("-" * 60)

print(
    df.groupby(["generator", "label_name"])["aspect_ratio"]
      .agg(["min", "median", "max"])
      .round(3)
)

# ---------------------------------------------------------
# 4. EXIF
# ---------------------------------------------------------

print("\n[4] EXIF PRESENCE")
print("-" * 60)

print(
    df.groupby(["generator", "label_name"])["path"]
      .count()
)

# ---------------------------------------------------------
# 5. Potential shortcut report
# ---------------------------------------------------------

print("\n[5] SHORTCUT CHECK")
print("-" * 60)

format_by_label = (
    df.groupby(["label_name", "format"])
      .size()
      .unstack(fill_value=0)
)

print(format_by_label)

print("\n⚠️ IMPORTANT:")

if set(df[df.label_name == "ai"]["format"].unique()) != \
   set(df[df.label_name == "nature"]["format"].unique()):

    print("❌ FORMAT DISTRIBUTIONS DIFFER STRONGLY.")
    print("   We MUST neutralize format/compression shortcuts.")

print("\nResolution ranges:")

print(
    df.groupby("label_name")[["width", "height"]]
      .agg(["min", "median", "max"])
)

print("\n" + "=" * 90)
print("BIAS ANALYSIS COMPLETE")
print("=" * 90)

In [ ]:
from pathlib import Path
import os

ROOT = Path("/kaggle/input/datasets/vtphatt2")
MJ = ROOT / "genimage-midjourney-part-1" / "part_aa"

print("Path:", MJ)
print("Exists:", MJ.exists())
print("Size GB:", round(MJ.stat().st_size / (1024**3), 2) if MJ.exists() else None)

# Inspect file type/header only
with open(MJ, "rb") as f:
    header = f.read(32)

print("Header:", header)

In [ ]:
!file "/kaggle/input/datasets/vtphatt2/genimage-midjourney-part-1/part_aa"

In [ ]:
!unzip -l "/kaggle/input/datasets/vtphatt2/genimage-midjourney-part-1/part_aa" 2>&1 | head -30

In [ ]:
from pathlib import Path
import pandas as pd
import time

START = time.time()

print("=" * 90)
print("SIGNALSCOPE — STEP 5: BUILD RAW DATASET MANIFEST")
print("=" * 90)

ROOT = Path("/kaggle/input/datasets/vtphatt2")

DATASETS = {
    "ADM": ROOT / "genimage-adm/GenImage/ADM/imagenet_ai_0508_adm/train",
    "BigGAN": ROOT / "genimage-biggan/BigGAN/imagenet_ai_0419_biggan/train",
    "GLIDE": ROOT / "genimage-glide/glide/imagenet_glide/train",
    "SD1.4": ROOT / "genimage-stable-diffusion-v1-4/GenImage/stable_diffusion_v_1_4/train",
    "SD1.5": ROOT / "genimage-stable-diffusion-v1-5/GenImage/stable_diffusion_v_1_5/train",
    "Wukong": ROOT / "genimage-wukong/GenImage/wukong/train",
}

ROWS = []

print("\n[START] Scanning dataset directories...\n")

for generator, base in DATASETS.items():

    print(f"▶ {generator}")
    generator_start = time.time()

    for label in ["ai", "nature"]:

        folder = base / label
        print(f"   ├─ Scanning {label}: {folder}")

        count = 0

        for p in folder.iterdir():
            if p.is_file():
                ROWS.append({
                    "path": str(p),
                    "generator": generator,
                    "label": 1 if label == "ai" else 0,
                    "label_name": label,
                })

                count += 1

                # Progress every 10,000 files
                if count % 10_000 == 0:
                    elapsed = time.time() - START
                    print(
                        f"   │  ⏳ {label}: {count:,} files | "
                        f"Total collected: {len(ROWS):,} | "
                        f"Elapsed: {elapsed:.1f}s"
                    )

        print(f"   └─ ✅ {label}: {count:,} files")

    print(
        f"   ✅ {generator} complete "
        f"({time.time() - generator_start:.1f}s)\n"
    )

print("=" * 90)
print("BUILDING DATAFRAME")
print("=" * 90)

df = pd.DataFrame(ROWS)

print(f"\n✅ DataFrame created")
print(f"Total images: {len(df):,}")

print("\nGenerator × Label:")
print(pd.crosstab(df["generator"], df["label_name"]))

print("\nTotal AI:", f"{(df['label'] == 1).sum():,}")
print("Total Real:", f"{(df['label'] == 0).sum():,}")

print("\n" + "=" * 90)
print("SAVING MANIFEST")
print("=" * 90)

OUT = Path("/kaggle/working/signalscope_audit")
OUT.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUT / "raw_manifest.csv"

print(f"💾 Saving to:\n{OUTPUT_FILE}")

df.to_csv(OUTPUT_FILE, index=False)

print("\n✅ Manifest saved successfully!")
print(f"📄 File size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"⏱️ Total execution time: {time.time() - START:.1f} seconds")

print("\n" + "=" * 90)
print("STEP 5 COMPLETE")
print("=" * 90)

In [9]:
from pathlib import Path
import pandas as pd
import time

START = time.time()

print("=" * 90)
print("SIGNALSCOPE — STEP 5B: BUILD CANDIDATE SUBSET")
print("=" * 90)

MANIFEST = Path("/kaggle/working/signalscope_audit/raw_manifest.csv")
OUT = Path("/kaggle/working/signalscope")
OUT.mkdir(parents=True, exist_ok=True)

if not MANIFEST.exists():
    raise FileNotFoundError(
        "raw_manifest.csv is missing. STOP here — do NOT rescan 1.9M files yet."
    )

print("\n[1/5] Loading raw manifest...")
df = pd.read_csv(MANIFEST)

print(f"✅ Loaded {len(df):,} rows")
print(f"⏱️ Elapsed: {time.time() - START:.1f}s")

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------

SEED = 42
AI_PER_GENERATOR = 5000
TOTAL_REAL = 30000

GENERATORS = [
    "ADM",
    "BigGAN",
    "GLIDE",
    "SD1.4",
    "SD1.5",
    "Wukong",
]

# ---------------------------------------------------------
# AI SAMPLE
# ---------------------------------------------------------

print("\n[2/5] Sampling AI images...")

ai_parts = []

for generator in GENERATORS:

    print(f"   ⏳ Sampling {generator}...")

    part = df[
        (df["generator"] == generator) &
        (df["label"] == 1)
    ]

    sampled = part.sample(
        n=AI_PER_GENERATOR,
        random_state=SEED
    )

    ai_parts.append(sampled)

    print(
        f"   ✅ {generator}: {len(sampled):,} selected "
        f"| available={len(part):,}"
    )

ai_df = pd.concat(ai_parts, ignore_index=True)

print(f"\n✅ Total AI selected: {len(ai_df):,}")

# ---------------------------------------------------------
# REAL SAMPLE
# ---------------------------------------------------------

print("\n[3/5] Preparing real-image pool...")

real_df = df[df["label"] == 0].copy()

print(f"   Raw real rows: {len(real_df):,}")

# GenImage may reuse the same ImageNet real image across generator folders.
# First deduplicate using filename.
real_df["filename"] = real_df["path"].map(
    lambda x: Path(x).name
)

before = len(real_df)

real_unique = real_df.drop_duplicates(
    subset=["filename"],
    keep="first"
)

print(f"   After filename dedup: {len(real_unique):,}")
print(f"   Potential repeated filenames removed: {before - len(real_unique):,}")

if len(real_unique) < TOTAL_REAL:
    raise RuntimeError(
        f"Only {len(real_unique):,} unique real filenames available."
    )

print("\n   ⏳ Sampling real images...")

real_sample = real_unique.sample(
    n=TOTAL_REAL,
    random_state=SEED
)

print(f"✅ Real selected: {len(real_sample):,}")

# ---------------------------------------------------------
# COMBINE
# ---------------------------------------------------------

print("\n[4/5] Combining candidate dataset...")

candidate = pd.concat(
    [ai_df, real_sample],
    ignore_index=True
)

candidate = candidate.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print(f"✅ Total candidate images: {len(candidate):,}")

print("\nClass distribution:")
print(candidate["label"].value_counts())

print("\nAI generator distribution:")
print(
    candidate[candidate["label"] == 1]
    ["generator"]
    .value_counts()
)

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

print("\n[5/5] Saving candidate manifest...")

output_file = OUT / "candidate_manifest_60k.csv"

candidate.to_csv(
    output_file,
    index=False
)

print(f"\n✅ Saved:")
print(output_file)

print(
    f"📄 Size: "
    f"{output_file.stat().st_size / (1024**2):.2f} MB"
)

print(
    f"⏱️ Total runtime: "
    f"{time.time() - START:.1f}s"
)

print("\n" + "=" * 90)
print("STEP 5B COMPLETE")
print("=" * 90)

SIGNALSCOPE — STEP 5B: BUILD CANDIDATE SUBSET

[1/5] Loading raw manifest...
✅ Loaded 1,933,463 rows
⏱️ Elapsed: 5.4s

[2/5] Sampling AI images...
   ⏳ Sampling ADM...
   ✅ ADM: 5,000 selected | available=162,000
   ⏳ Sampling BigGAN...
   ✅ BigGAN: 5,000 selected | available=162,000
   ⏳ Sampling GLIDE...
   ✅ GLIDE: 5,000 selected | available=162,000
   ⏳ Sampling SD1.4...
   ✅ SD1.4: 5,000 selected | available=161,997
   ⏳ Sampling SD1.5...
   ✅ SD1.5: 5,000 selected | available=166,000
   ⏳ Sampling Wukong...
   ✅ Wukong: 5,000 selected | available=162,000

✅ Total AI selected: 30,000

[3/5] Preparing real-image pool...
   Raw real rows: 957,466
   After filename dedup: 957,466
   Potential repeated filenames removed: 0

   ⏳ Sampling real images...
✅ Real selected: 30,000

[4/5] Combining candidate dataset...
✅ Total candidate images: 60,000

Class distribution:
label
1    30000
0    30000
Name: count, dtype: int64

AI generator distribution:
generator
GLIDE     5000
BigGAN    500

In [10]:
from pathlib import Path
from PIL import Image
import pandas as pd
import hashlib
import time

START = time.time()

print("=" * 90)
print("SIGNALSCOPE — STEP 6: VERIFY CANDIDATE DATASET")
print("=" * 90)

MANIFEST = Path("/kaggle/working/signalscope/candidate_manifest_60k.csv")
OUT = Path("/kaggle/working/signalscope")
OUTPUT = OUT / "verified_manifest_60k.csv"

print("\n[1/4] Loading candidate manifest...")

df = pd.read_csv(MANIFEST)

print(f"✅ Loaded {len(df):,} rows")
print(f"⏱️ Elapsed: {time.time() - START:.1f}s")

# ---------------------------------------------------------
# HASH + CORRUPTION CHECK
# ---------------------------------------------------------

print("\n[2/4] Checking files + computing SHA256...")
print("This opens each of the 60,000 selected images once.\n")

results = []

total = len(df)

for idx, row in df.iterrows():

    path = Path(row["path"])

    status = "ok"
    sha256 = None
    width = None
    height = None

    try:
        # -----------------------------------------
        # Verify image readability
        # -----------------------------------------
        with Image.open(path) as img:
            width, height = img.size
            img.verify()

        # -----------------------------------------
        # SHA256 exact-content hash
        # -----------------------------------------
        h = hashlib.sha256()

        with open(path, "rb") as f:
            while True:
                chunk = f.read(1024 * 1024)

                if not chunk:
                    break

                h.update(chunk)

        sha256 = h.hexdigest()

    except Exception as e:
        status = f"corrupt:{type(e).__name__}"

    results.append({
        "sha256": sha256,
        "verify_status": status,
        "width": width,
        "height": height,
    })

    # Progress every 1,000 images
    done = idx + 1

    if done % 1000 == 0 or done == total:

        elapsed = time.time() - START
        rate = done / elapsed if elapsed > 0 else 0

        print(
            f"⏳ {done:,}/{total:,} checked "
            f"({done/total*100:.1f}%) | "
            f"{rate:.1f} images/sec | "
            f"Elapsed: {elapsed/60:.1f} min"
        )

# ---------------------------------------------------------
# ATTACH RESULTS
# ---------------------------------------------------------

print("\n[3/4] Analysing verification results...")

result_df = pd.DataFrame(results)

df = pd.concat(
    [df.reset_index(drop=True), result_df],
    axis=1
)

corrupt = df[df["verify_status"] != "ok"]

print(f"\nCorrupted/unreadable: {len(corrupt):,}")

# ---------------------------------------------------------
# EXACT DUPLICATES
# ---------------------------------------------------------

valid = df[
    (df["verify_status"] == "ok") &
    (df["sha256"].notna())
].copy()

duplicate_mask = valid.duplicated(
    subset=["sha256"],
    keep="first"
)

duplicates = valid[duplicate_mask]

print(f"Exact duplicate rows: {len(duplicates):,}")

# Number of hashes occurring multiple times
duplicate_groups = (
    valid.groupby("sha256")
    .size()
    .sort_values(ascending=False)
)

duplicate_groups = duplicate_groups[
    duplicate_groups > 1
]

print(f"Duplicate hash groups: {len(duplicate_groups):,}")

# ---------------------------------------------------------
# CHECK LABEL COLLISIONS
# ---------------------------------------------------------

collision = (
    valid.groupby("sha256")["label"]
    .nunique()
)

collision = collision[
    collision > 1
]

print(f"⚠️ Same image appearing as BOTH AI and real: {len(collision):,}")

# ---------------------------------------------------------
# CLEAN
# ---------------------------------------------------------

print("\n[4/4] Creating verified manifest...")

clean = valid.drop_duplicates(
    subset=["sha256"],
    keep="first"
).copy()

clean.reset_index(drop=True, inplace=True)

clean.to_csv(
    OUTPUT,
    index=False
)

print("\nFinal counts:")
print(clean["label"].value_counts())

print("\nAI generators:")
print(
    clean[clean["label"] == 1]
    ["generator"]
    .value_counts()
)

print("\n✅ Saved:")
print(OUTPUT)

print(
    f"📄 Size: "
    f"{OUTPUT.stat().st_size / (1024**2):.2f} MB"
)

print(
    f"⏱️ Total runtime: "
    f"{(time.time() - START)/60:.1f} minutes"
)

print("\n" + "=" * 90)
print("STEP 6 COMPLETE")
print("=" * 90)

SIGNALSCOPE — STEP 6: VERIFY CANDIDATE DATASET

[1/4] Loading candidate manifest...
✅ Loaded 60,000 rows
⏱️ Elapsed: 0.2s

[2/4] Checking files + computing SHA256...
This opens each of the 60,000 selected images once.

⏳ 1,000/60,000 checked (1.7%) | 86.8 images/sec | Elapsed: 0.2 min
⏳ 2,000/60,000 checked (3.3%) | 90.2 images/sec | Elapsed: 0.4 min
⏳ 3,000/60,000 checked (5.0%) | 91.9 images/sec | Elapsed: 0.5 min
⏳ 4,000/60,000 checked (6.7%) | 92.3 images/sec | Elapsed: 0.7 min
⏳ 5,000/60,000 checked (8.3%) | 92.2 images/sec | Elapsed: 0.9 min
⏳ 6,000/60,000 checked (10.0%) | 92.2 images/sec | Elapsed: 1.1 min
⏳ 7,000/60,000 checked (11.7%) | 92.0 images/sec | Elapsed: 1.3 min
⏳ 8,000/60,000 checked (13.3%) | 91.4 images/sec | Elapsed: 1.5 min
⏳ 9,000/60,000 checked (15.0%) | 89.4 images/sec | Elapsed: 1.7 min
⏳ 10,000/60,000 checked (16.7%) | 88.3 images/sec | Elapsed: 1.9 min
⏳ 11,000/60,000 checked (18.3%) | 87.8 images/sec | Elapsed: 2.1 min
⏳ 12,000/60,000 checked (20.0%) | 85

In [11]:
from pathlib import Path
import pandas as pd
import time

START = time.time()

print("=" * 90)
print("SIGNALSCOPE — STEP 7: TRAIN / VALIDATION / UNSEEN SPLIT")
print("=" * 90)

INPUT = Path("/kaggle/working/signalscope/verified_manifest_60k.csv")
OUT = Path("/kaggle/working/signalscope/splits")
OUT.mkdir(parents=True, exist_ok=True)

UNSEEN_GENERATOR = "Wukong"
SEED = 42
TRAIN_RATIO = 0.80

print("\n[1/6] Loading verified manifest...")

if not INPUT.exists():
    raise FileNotFoundError(
        f"Missing file:\n{INPUT}\n\n"
        "If using a new notebook, upload/mount the verified manifest first."
    )

df = pd.read_csv(INPUT)

print(f"✅ Loaded: {len(df):,} images")
print(f"⏱️ Elapsed: {time.time() - START:.1f}s")

print("\nGenerator distribution:")
print(pd.crosstab(df["generator"], df["label_name"]))

# ---------------------------------------------------------
# HOLD OUT UNSEEN GENERATOR
# ---------------------------------------------------------

print(f"\n[2/6] Holding out generator: {UNSEEN_GENERATOR}")

unseen = df[
    df["generator"] == UNSEEN_GENERATOR
].copy()

seen = df[
    df["generator"] != UNSEEN_GENERATOR
].copy()

print(f"✅ Seen-generator pool:   {len(seen):,}")
print(f"✅ Unseen-generator pool: {len(unseen):,}")

# ---------------------------------------------------------
# STRATIFIED TRAIN / NORMAL VALIDATION
# ---------------------------------------------------------

print("\n[3/6] Creating stratified train/normal-validation split...")

train_parts = []
val_parts = []

for generator in sorted(seen["generator"].unique()):

    print(f"   ⏳ Processing {generator}...")

    for label in [0, 1]:

        part = seen[
            (seen["generator"] == generator) &
            (seen["label"] == label)
        ].copy()

        n_train = int(len(part) * TRAIN_RATIO)

        train_part = part.sample(
            n=n_train,
            random_state=SEED
        )

        val_part = part.drop(
            train_part.index
        )

        train_parts.append(train_part)
        val_parts.append(val_part)

        print(
            f"      {'AI' if label == 1 else 'Real'}: "
            f"train={len(train_part):,}, "
            f"val={len(val_part):,}"
        )

train_df = pd.concat(
    train_parts,
    ignore_index=True
)

normal_val_df = pd.concat(
    val_parts,
    ignore_index=True
)

print("\n✅ Train:", f"{len(train_df):,}")
print("✅ Normal validation:", f"{len(normal_val_df):,}")

# ---------------------------------------------------------
# SHUFFLE
# ---------------------------------------------------------

print("\n[4/6] Shuffling splits...")

train_df = train_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

normal_val_df = normal_val_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

unseen = unseen.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print("✅ Shuffling complete")

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

print("\n[5/6] Saving split manifests...")

train_path = OUT / "train.csv"
normal_val_path = OUT / "normal_val.csv"
unseen_path = OUT / "unseen_wukong.csv"

train_df.to_csv(train_path, index=False)
normal_val_df.to_csv(normal_val_path, index=False)
unseen.to_csv(unseen_path, index=False)

print(f"✅ Train:          {train_path}")
print(f"✅ Normal Val:     {normal_val_path}")
print(f"✅ Unseen Wukong:  {unseen_path}")

# ---------------------------------------------------------
# FINAL VERIFICATION
# ---------------------------------------------------------

print("\n[6/6] Verifying split integrity...")

print("\nTRAIN:")
print(pd.crosstab(train_df["generator"], train_df["label_name"]))

print("\nNORMAL VALIDATION:")
print(pd.crosstab(normal_val_df["generator"], normal_val_df["label_name"]))

print("\nUNSEEN:")
print(pd.crosstab(unseen["generator"], unseen["label_name"]))

# Check generator leakage
assert UNSEEN_GENERATOR not in train_df["generator"].unique()
assert UNSEEN_GENERATOR not in normal_val_df["generator"].unique()

# Check path overlap
train_paths = set(train_df["path"])
val_paths = set(normal_val_df["path"])
unseen_paths = set(unseen["path"])

assert not train_paths & val_paths
assert not train_paths & unseen_paths
assert not val_paths & unseen_paths

print("\n✅ No path leakage detected")

print("\nFinal sizes:")
print(f"Train:          {len(train_df):,}")
print(f"Normal Val:     {len(normal_val_df):,}")
print(f"Unseen Wukong:  {len(unseen):,}")

print(
    f"\n⏱️ Total runtime: "
    f"{time.time() - START:.1f}s"
)

print("\n" + "=" * 90)
print("STEP 7 COMPLETE")
print("=" * 90)

SIGNALSCOPE — STEP 7: TRAIN / VALIDATION / UNSEEN SPLIT

[1/6] Loading verified manifest...
✅ Loaded: 59,993 images
⏱️ Elapsed: 0.5s

Generator distribution:
label_name    ai  nature
generator               
ADM         5000    4988
BigGAN      5000    5028
GLIDE       5000    5047
SD1.4       5000    5129
SD1.5       4999    4796
Wukong      5000    5006

[2/6] Holding out generator: Wukong
✅ Seen-generator pool:   49,987
✅ Unseen-generator pool: 10,006

[3/6] Creating stratified train/normal-validation split...
   ⏳ Processing ADM...
      Real: train=3,990, val=998
      AI: train=4,000, val=1,000
   ⏳ Processing BigGAN...
      Real: train=4,022, val=1,006
      AI: train=4,000, val=1,000
   ⏳ Processing GLIDE...
      Real: train=4,037, val=1,010
      AI: train=4,000, val=1,000
   ⏳ Processing SD1.4...
      Real: train=4,103, val=1,026
      AI: train=4,000, val=1,000
   ⏳ Processing SD1.5...
      Real: train=3,836, val=960
      AI: train=3,999, val=1,000

✅ Train: 39,987
✅ No

In [12]:
import torch
from torchvision import transforms
from PIL import Image
import random
import io

IMAGE_SIZE = 224

class RandomJPEG:
    def __init__(self, p=0.35, quality_min=40, quality_max=100):
        self.p = p
        self.quality_min = quality_min
        self.quality_max = quality_max

    def __call__(self, img):
        if random.random() > self.p:
            return img

        quality = random.randint(
            self.quality_min,
            self.quality_max
        )

        buffer = io.BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=quality
        )

        buffer.seek(0)

        return Image.open(buffer).convert("RGB")


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.75, 1.0),
        ratio=(0.80, 1.25)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    RandomJPEG(
        p=0.35,
        quality_min=40,
        quality_max=100
    ),

    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(0.1, 1.0)
        )
    ], p=0.12),

    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.10,
            contrast=0.10,
            saturation=0.10,
            hue=0.02
        )
    ], p=0.15),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([
    transforms.Resize(256),

    transforms.CenterCrop(
        IMAGE_SIZE
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


print("=" * 80)
print("SIGNALSCOPE — STEP 8 PREPROCESSING")
print("=" * 80)

print("\n✅ IMAGE_SIZE:", IMAGE_SIZE)

print("\nTraining transform:")
print(train_transform)

print("\nValidation transform:")
print(val_transform)

print("\n✅ STEP 8 COMPLETE")

SIGNALSCOPE — STEP 8 PREPROCESSING

✅ IMAGE_SIZE: 224

Training transform:
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.75, 1.0), ratio=(0.8, 1.25), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomApply(
    p=0.12
    GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.0))
)
    RandomApply(
    p=0.15
    ColorJitter(brightness=(0.9, 1.1), contrast=(0.9, 1.1), saturation=(0.9, 1.1), hue=(-0.02, 0.02))
)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Validation transform:
Compose(
    Resize(size=256, interpolation=bilinear, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

✅ STEP 8 COMPLETE


In [4]:
from pathlib import Path
import pandas as pd
import time

START = time.time()

print("=" * 90)
print("SIGNALSCOPE — STEP 7 REBUILD: TRAIN / VAL / UNSEEN SPLITS")
print("=" * 90)

# =========================================================
# FIND CANDIDATE MANIFEST
# =========================================================

print("\n[1/6] Locating candidate_manifest_60k.csv...")

possible_paths = [
    Path("/kaggle/input/candidate_manifest_60k/candidate_manifest_60k.csv"),
    Path("/kaggle/input/signalscope/candidate_manifest_60k.csv"),
    Path("/kaggle/working/signalscope/candidate_manifest_60k.csv"),
    Path("/kaggle/input/datasets/shahsujal10/candidate-manifest-60k/candidate_manifest_60k.csv")
]

INPUT = None

for p in possible_paths:
    print(f"   Checking: {p}")
    if p.exists():
        INPUT = p
        break

if INPUT is None:
    # Search only /kaggle/input, NOT the image directories
    matches = list(Path("/kaggle/input").glob("**/candidate_manifest_60k.csv"))

    if matches:
        INPUT = matches[0]

if INPUT is None:
    raise FileNotFoundError(
        "❌ candidate_manifest_60k.csv not found.\n"
        "Please attach/upload it as a Kaggle Dataset."
    )

print(f"\n✅ Found:\n{INPUT}")

# =========================================================
# LOAD
# =========================================================

print("\n[2/6] Loading candidate manifest...")

df = pd.read_csv(INPUT)

print(f"✅ Loaded {len(df):,} images")
print(f"⏱️ Elapsed: {time.time() - START:.1f}s")

print("\nGenerator × Label:")
print(pd.crosstab(df["generator"], df["label_name"]))

# =========================================================
# SETTINGS
# =========================================================

SEED = 42
TRAIN_RATIO = 0.80
UNSEEN_GENERATOR = "Wukong"

print("\n[3/6] Split configuration:")
print(f"   Unseen generator: {UNSEEN_GENERATOR}")
print(f"   Train ratio:      {TRAIN_RATIO}")
print(f"   Random seed:      {SEED}")

# =========================================================
# HOLD OUT WUKONG
# =========================================================

print("\n[4/6] Creating splits...")

unseen_df = df[
    df["generator"] == UNSEEN_GENERATOR
].copy()

seen_df = df[
    df["generator"] != UNSEEN_GENERATOR
].copy()

print(f"   Seen pool:   {len(seen_df):,}")
print(f"   Unseen pool: {len(unseen_df):,}")

train_parts = []
val_parts = []

for generator in sorted(seen_df["generator"].unique()):

    print(f"\n   ▶ {generator}")

    for label in [0, 1]:

        part = seen_df[
            (seen_df["generator"] == generator) &
            (seen_df["label"] == label)
        ].copy()

        n_train = int(len(part) * TRAIN_RATIO)

        train_part = part.sample(
            n=n_train,
            random_state=SEED
        )

        val_part = part.drop(train_part.index)

        train_parts.append(train_part)
        val_parts.append(val_part)

        print(
            f"      {'AI' if label == 1 else 'Real'}: "
            f"train={len(train_part):,} | "
            f"val={len(val_part):,}"
        )

train_df = pd.concat(train_parts, ignore_index=True)
normal_val_df = pd.concat(val_parts, ignore_index=True)

# Shuffle
train_df = train_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

normal_val_df = normal_val_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

unseen_df = unseen_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print("\n✅ Splits created")

# =========================================================
# SAVE
# =========================================================

print("\n[5/6] Saving split manifests...")

OUT = Path("/kaggle/working/signalscope/splits")
OUT.mkdir(parents=True, exist_ok=True)

train_path = OUT / "train.csv"
val_path = OUT / "normal_val.csv"
unseen_path = OUT / "unseen_wukong.csv"

train_df.to_csv(train_path, index=False)
normal_val_df.to_csv(val_path, index=False)
unseen_df.to_csv(unseen_path, index=False)

print(f"✅ Train:         {train_path}")
print(f"✅ Normal Val:    {val_path}")
print(f"✅ Unseen Wukong: {unseen_path}")

# =========================================================
# VERIFY
# =========================================================

print("\n[6/6] Verifying split integrity...")

print("\nTRAIN:")
print(pd.crosstab(train_df["generator"], train_df["label_name"]))

print("\nNORMAL VALIDATION:")
print(pd.crosstab(normal_val_df["generator"], normal_val_df["label_name"]))

print("\nUNSEEN:")
print(pd.crosstab(unseen_df["generator"], unseen_df["label_name"]))

# No generator leakage
assert UNSEEN_GENERATOR not in train_df["generator"].unique()
assert UNSEEN_GENERATOR not in normal_val_df["generator"].unique()

# No path leakage
train_paths = set(train_df["path"])
val_paths = set(normal_val_df["path"])
unseen_paths = set(unseen_df["path"])

assert not train_paths & val_paths
assert not train_paths & unseen_paths
assert not val_paths & unseen_paths

print("\n✅ No path leakage")
print("✅ Wukong completely unseen during training")

print("\nFINAL:")
print(f"   Train:         {len(train_df):,}")
print(f"   Normal Val:    {len(normal_val_df):,}")
print(f"   Unseen Wukong: {len(unseen_df):,}")

print(f"\n⏱️ Total runtime: {time.time() - START:.1f}s")

print("\n" + "=" * 90)
print("STEP 7 REBUILD COMPLETE")
print("=" * 90)

SIGNALSCOPE — STEP 7 REBUILD: TRAIN / VAL / UNSEEN SPLITS

[1/6] Locating candidate_manifest_60k.csv...
   Checking: /kaggle/input/candidate_manifest_60k/candidate_manifest_60k.csv
   Checking: /kaggle/input/signalscope/candidate_manifest_60k.csv
   Checking: /kaggle/working/signalscope/candidate_manifest_60k.csv
   Checking: /kaggle/input/datasets/shahsujal10/candidate-manifest-60k/candidate_manifest_60k.csv

✅ Found:
/kaggle/input/datasets/shahsujal10/candidate-manifest-60k/candidate_manifest_60k.csv

[2/6] Loading candidate manifest...
✅ Loaded 60,000 images
⏱️ Elapsed: 0.2s

Generator × Label:
label_name    ai  nature
generator               
ADM         5000    4989
BigGAN      5000    5029
GLIDE       5000    5048
SD1.4       5000    5130
SD1.5       5000    4797
Wukong      5000    5007

[3/6] Split configuration:
   Unseen generator: Wukong
   Train ratio:      0.8
   Random seed:      42

[4/6] Creating splits...
   Seen pool:   49,993
   Unseen pool: 10,007

   ▶ ADM
      Re

In [5]:
# =============================================================================
# SIGNALSCOPE — STEP 8: PREPROCESSING
# =============================================================================

import torch
import torchvision
from torchvision import transforms
from PIL import Image
import random
import io

print("=" * 90)
print("SIGNALSCOPE — STEP 8: PREPROCESSING")
print("=" * 90)

# -------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------
IMAGE_SIZE = 224

print(f"\n[1/4] Configuration")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")

# -------------------------------------------------------------------------
# 2. Custom JPEG augmentation
# -------------------------------------------------------------------------
class RandomJPEG:
    def __init__(self, p=0.35, quality_min=40, quality_max=100):
        self.p = p
        self.quality_min = quality_min
        self.quality_max = quality_max

    def __call__(self, img):
        if random.random() > self.p:
            return img

        quality = random.randint(
            self.quality_min,
            self.quality_max
        )

        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)

        return Image.open(buffer).convert("RGB")


# -------------------------------------------------------------------------
# 3. Training transform
# -------------------------------------------------------------------------
print("\n[2/4] Creating training transform...")

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.75, 1.0),
        ratio=(0.80, 1.25)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    RandomJPEG(
        p=0.35,
        quality_min=40,
        quality_max=100
    ),

    transforms.RandomApply(
        [
            transforms.GaussianBlur(
                kernel_size=3,
                sigma=(0.1, 1.0)
            )
        ],
        p=0.12
    ),

    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.10,
                contrast=0.10,
                saturation=0.10,
                hue=0.02
            )
        ],
        p=0.15
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Training transform ready")


# -------------------------------------------------------------------------
# 4. Validation transform
# -------------------------------------------------------------------------
print("\n[3/4] Creating validation transform...")

val_transform = transforms.Compose([
    transforms.Resize(256),

    transforms.CenterCrop(IMAGE_SIZE),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Validation transform ready")


# -------------------------------------------------------------------------
# 5. Quick transform sanity test
# -------------------------------------------------------------------------
print("\n[4/4] Running sanity test...")

test_img = Image.new("RGB", (512, 512), color="white")

train_out = train_transform(test_img)
val_out = val_transform(test_img)

print(f"Train output shape: {tuple(train_out.shape)}")
print(f"Val output shape:   {tuple(val_out.shape)}")

assert train_out.shape == (3, 224, 224)
assert val_out.shape == (3, 224, 224)

print("✅ Transform sanity test passed")

print("\n" + "=" * 90)
print("STEP 8 COMPLETE")
print("=" * 90)

SIGNALSCOPE — STEP 8: PREPROCESSING

[1/4] Configuration
Image size: 224x224

[2/4] Creating training transform...
✅ Training transform ready

[3/4] Creating validation transform...
✅ Validation transform ready

[4/4] Running sanity test...
Train output shape: (3, 224, 224)
Val output shape:   (3, 224, 224)
✅ Transform sanity test passed

STEP 8 COMPLETE


In [6]:
# =============================================================================
# SIGNALSCOPE — STEP 9: DATALOADER + GPU SANITY CHECK
# =============================================================================

import os
import time
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

print("=" * 90)
print("SIGNALSCOPE — STEP 9: DATALOADER + GPU SANITY CHECK")
print("=" * 90)

# -------------------------------------------------------------------------
# 1. Paths
# -------------------------------------------------------------------------
print("\n[1/5] Checking split files...")

TRAIN_CSV = "/kaggle/working/signalscope/splits/train.csv"
VAL_CSV = "/kaggle/working/signalscope/splits/normal_val.csv"
UNSEEN_CSV = "/kaggle/working/signalscope/splits/unseen_wukong.csv"

for path in [TRAIN_CSV, VAL_CSV, UNSEEN_CSV]:
    print(f"Checking: {path}")
    assert os.path.exists(path), f"Missing: {path}"

print("✅ All split files found")


# -------------------------------------------------------------------------
# 2. Dataset
# -------------------------------------------------------------------------
print("\n[2/5] Creating dataset class...")

class SignalScopeDataset(Dataset):

    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["path"]
        label = float(row["label"])

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)


print("✅ Dataset class ready")


# -------------------------------------------------------------------------
# 3. Create datasets
# -------------------------------------------------------------------------
print("\n[3/5] Loading datasets...")

train_dataset = SignalScopeDataset(
    TRAIN_CSV,
    transform=train_transform
)

val_dataset = SignalScopeDataset(
    VAL_CSV,
    transform=val_transform
)

unseen_dataset = SignalScopeDataset(
    UNSEEN_CSV,
    transform=val_transform
)

print(f"Train samples:         {len(train_dataset):,}")
print(f"Normal validation:     {len(val_dataset):,}")
print(f"Unseen Wukong:         {len(unseen_dataset):,}")


# -------------------------------------------------------------------------
# 4. DataLoaders
# -------------------------------------------------------------------------
print("\n[4/5] Creating DataLoaders...")

BATCH_SIZE = 16
NUM_WORKERS = 2

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

unseen_loader = DataLoader(
    unseen_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory
)

print("✅ DataLoaders created")
print(f"Batch size: {BATCH_SIZE}")
print(f"Workers:    {NUM_WORKERS}")


# -------------------------------------------------------------------------
# 5. GPU + batch sanity check
# -------------------------------------------------------------------------
print("\n[5/5] GPU and batch sanity check...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"CUDA:   {torch.version.cuda}")
else:
    print("⚠️ CUDA NOT AVAILABLE")


print("\n▶ Loading one training batch...")

start = time.time()

images, labels = next(iter(train_loader))

elapsed = time.time() - start

print(f"Batch load time: {elapsed:.2f}s")
print(f"Images shape:    {tuple(images.shape)}")
print(f"Labels shape:    {tuple(labels.shape)}")
print(f"Label values:    {labels[:10].tolist()}")

assert images.shape == (BATCH_SIZE, 3, 224, 224)
assert labels.shape == (BATCH_SIZE,)

print("✅ Training batch sanity check passed")

print("\n▶ Loading one validation batch...")

images_val, labels_val = next(iter(val_loader))

print(f"Val images shape: {tuple(images_val.shape)}")
print(f"Val labels shape: {tuple(labels_val.shape)}")

assert images_val.shape == (BATCH_SIZE, 3, 224, 224)

print("✅ Validation batch sanity check passed")


print("\n" + "=" * 90)
print("STEP 9 COMPLETE")
print("=" * 90)

SIGNALSCOPE — STEP 9: DATALOADER + GPU SANITY CHECK

[1/5] Checking split files...
Checking: /kaggle/working/signalscope/splits/train.csv
Checking: /kaggle/working/signalscope/splits/normal_val.csv
Checking: /kaggle/working/signalscope/splits/unseen_wukong.csv
✅ All split files found

[2/5] Creating dataset class...
✅ Dataset class ready

[3/5] Loading datasets...
Train samples:         39,993
Normal validation:     10,000
Unseen Wukong:         10,007

[4/5] Creating DataLoaders...
✅ DataLoaders created
Batch size: 16
Workers:    2

[5/5] GPU and batch sanity check...
Device: cuda
GPU:    Tesla T4
CUDA:   12.8

▶ Loading one training batch...
Batch load time: 0.98s
Images shape:    (16, 3, 224, 224)
Labels shape:    (16,)
Label values:    [1.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0]
✅ Training batch sanity check passed

▶ Loading one validation batch...
Val images shape: (16, 3, 224, 224)
Val labels shape: (16,)
✅ Validation batch sanity check passed

STEP 9 COMPLETE


In [ ]:
# =============================================================================
# SIGNALSCOPE — STEP 10: E1 CONVNEXT-TINY BASELINE
# =============================================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)

print("=" * 90)
print("SIGNALSCOPE — STEP 10: E1 CONVNEXT-TINY")
print("=" * 90)

# -------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 3
LR = 1e-4

SAVE_DIR = "/kaggle/working/signalscope/experiments/E1"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"\n[1/6] Configuration")
print(f"Device:     {DEVICE}")
print(f"Epochs:     {EPOCHS}")
print(f"Learning rate: {LR}")
print(f"Save dir:   {SAVE_DIR}")

if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")


# -------------------------------------------------------------------------
# 2. Model
# -------------------------------------------------------------------------
print("\n[2/6] Loading pretrained ConvNeXt-Tiny...")

weights = ConvNeXt_Tiny_Weights.DEFAULT
model = convnext_tiny(weights=weights)

# Replace classifier
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, 1)

model = model.to(DEVICE)

print("✅ ConvNeXt-Tiny loaded")
print(f"Classifier input features: {in_features}")


# -------------------------------------------------------------------------
# 3. Loss / optimizer / AMP
# -------------------------------------------------------------------------
print("\n[3/6] Setting training components...")

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scaler = torch.amp.GradScaler("cuda") if torch.cuda.is_available() else None

print("✅ Training components ready")


# -------------------------------------------------------------------------
# 4. Evaluation function
# -------------------------------------------------------------------------
def evaluate(model, loader, name):

    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(DEVICE, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda",
                enabled=torch.cuda.is_available()
            ):
                logits = model(images).squeeze(1)

            probs = torch.sigmoid(logits)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = (y_prob >= 0.5).astype(int)

    auc = roc_auc_score(y_true, y_prob)
    f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)

    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) else 0

    print(f"\n{name}")
    print("-" * 60)
    print(f"ROC-AUC:   {auc:.4f}")
    print(f"Macro-F1:   {f1:.4f}")
    print(f"Accuracy:   {acc:.4f}")
    print(f"Precision:  {precision:.4f}")
    print(f"Recall:     {recall:.4f}")
    print(f"FPR:        {fpr:.4f}")
    print("Confusion Matrix:")
    print(cm)

    return {
        "auc": auc,
        "f1": f1,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "fpr": fpr
    }


# -------------------------------------------------------------------------
# 5. Training
# -------------------------------------------------------------------------
print("\n[4/6] Starting training...")
print("=" * 90)

best_val_auc = 0.0
history = []

for epoch in range(EPOCHS):

    epoch_start = time.time()

    model.train()

    running_loss = 0.0
    total = 0

    for batch_idx, (images, labels) in enumerate(train_loader):

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        batch_size = images.size(0)

        running_loss += loss.item() * batch_size
        total += batch_size

        if batch_idx == 0 or (batch_idx + 1) % 250 == 0:
            print(
                f"Epoch {epoch+1}/{EPOCHS} | "
                f"Batch {batch_idx+1}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = running_loss / total

    # Normal validation
    val_metrics = evaluate(
        model,
        val_loader,
        f"Epoch {epoch+1} — NORMAL VALIDATION"
    )

    # Unseen Wukong
    unseen_metrics = evaluate(
        model,
        unseen_loader,
        f"Epoch {epoch+1} — UNSEEN WUKONG"
    )

    epoch_time = time.time() - epoch_start

    print(
        f"\nEpoch {epoch+1} COMPLETE | "
        f"Train Loss: {train_loss:.4f} | "
        f"Normal AUC: {val_metrics['auc']:.4f} | "
        f"Unseen AUC: {unseen_metrics['auc']:.4f} | "
        f"Time: {epoch_time/60:.2f} min"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "normal_auc": val_metrics["auc"],
        "normal_f1": val_metrics["f1"],
        "unseen_auc": unseen_metrics["auc"],
        "unseen_f1": unseen_metrics["f1"],
        "unseen_accuracy": unseen_metrics["accuracy"]
    })

    # Save based on UNSEEN AUC
    if unseen_metrics["auc"] > best_val_auc:

        best_val_auc = unseen_metrics["auc"]

        checkpoint_path = os.path.join(
            SAVE_DIR,
            "best_convnext_tiny.pt"
        )

        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": epoch + 1,
            "unseen_auc": best_val_auc
        }, checkpoint_path)

        print(f"✅ NEW BEST — Unseen AUC: {best_val_auc:.4f}")
        print(f"Checkpoint saved: {checkpoint_path}")


# -------------------------------------------------------------------------
# 6. Save experiment results
# -------------------------------------------------------------------------
print("\n[5/6] Saving experiment history...")

history_df = pd.DataFrame(history)

history_path = os.path.join(
    SAVE_DIR,
    "training_history.csv"
)

history_df.to_csv(history_path, index=False)

print(f"✅ History saved: {history_path}")

print("\n[6/6] Final E1 summary")
print("=" * 90)
print(history_df.to_string(index=False))

print("\n" + "=" * 90)
print("STEP 10 COMPLETE")
print("=" * 90)
print(f"BEST UNSEEN WUKONG AUC: {best_val_auc:.4f}")
print("=" * 90)